# pyaegean neural-lemmatizer spike — GreBERTa + edit-tree head

Goal: beat stanza/CLTK on the hardest column, **unseen-form lemma** (pure-Python baseline 40.3%, stanza 62.8%) — by classifying the **same edit-tree label set** the pure-Python lemmatizer uses, but from a fine-tuned Ancient-Greek transformer.

**Loop:** set runtime to a **GPU** (H100 ideal; T4 fine) → Run all → it prints the dev lemma score *in Colab* → download `spike_model.zip` → send it back for the local eval.

Precision auto-detects: **bf16 + TF32 + fused AdamW + batch 64 on H100/A100**, fp16 + batch 16 on a T4. Encoder: `bowphs/GreBerta` (Apache-2.0). torch/transformers are used **only here**; production inference is onnxruntime-only. We ship **fp32** (see the export note).

In [ ]:
!nvidia-smi -L  # MUST list a GPU; if it says 'command not found' you're on CPU — fix the runtime
%pip -q install 'transformers>=4.46' 'datasets>=2.19' 'optimum[onnxruntime]>=1.20' 'accelerate>=0.30' onnx onnxruntime

## 0 · Detect GPU + precision

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > GPU, then reconnect.'
gpu = torch.cuda.get_device_name(0)
USE_BF16 = torch.cuda.is_bf16_supported()  # Ampere/Hopper+
BS = 64 if USE_BF16 else 16
print(f'torch {torch.__version__} | CUDA {torch.version.cuda} | GPU {gpu}')
print(f'bf16={USE_BF16} -> batch_size={BS}, precision={"bf16+tf32" if USE_BF16 else "fp16"}')

## 1 · Upload the data bundle
Upload **`spike_data.zip`** (`train.jsonl`, `dev.jsonl`, `labels.json`).

In [ ]:
import json, zipfile, pathlib
from google.colab import files
up = files.upload()  # pick spike_data.zip
zname = next(n for n in up if n.endswith('.zip'))
zipfile.ZipFile(zname).extractall('.')
DATA = pathlib.Path('data') if pathlib.Path('data/labels.json').exists() else pathlib.Path('.')
labels = json.loads((DATA / 'labels.json').read_text(encoding='utf-8'))['trees']
id2label = {i: k for i, k in enumerate(labels)}
label2id = {k: i for i, k in enumerate(labels)}
print('edit-tree labels:', len(labels))

## 2 · Load the training data

In [ ]:
from datasets import Dataset
def read_jsonl(p):
    return [json.loads(l) for l in open(p, encoding='utf-8')]
train_rows = read_jsonl(DATA / 'train.jsonl')
ds = Dataset.from_list([{'tokens': r['tokens'], 'tags': r['labels']} for r in train_rows])
print(ds)

## 3 · Tokenizer + model (`bowphs/GreBerta`)

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
MODEL = 'bowphs/GreBerta'
tokenizer = AutoTokenizer.from_pretrained(MODEL, add_prefix_space=True)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL, num_labels=len(labels), id2label=id2label, label2id=label2id)

## 4 · Tokenize + align labels to the first sub-token of each word
Label only the first sub-token of each word, `-100` (ignored by the loss) elsewhere.

In [ ]:
def align(batch):
    enc = tokenizer(batch['tokens'], is_split_into_words=True, truncation=True,
                    max_length=256)
    out = []
    for i, tags in enumerate(batch['tags']):
        word_ids = enc.word_ids(i)
        prev, row = None, []
        for wid in word_ids:
            if wid is None:
                row.append(-100)
            elif wid != prev:
                row.append(tags[wid])
            else:
                row.append(-100)
            prev = wid
        out.append(row)
    enc['labels'] = out
    return enc
tok_ds = ds.map(align, batched=True, remove_columns=ds.column_names)
tok_ds = tok_ds.train_test_split(test_size=0.02, seed=0)
print(tok_ds)

## 5 · Fine-tune (H100-tuned; keeps the best epoch by dev token-accuracy)
Watch `tok_acc` climb each epoch — if it stays near the identity rate, training didn't take (check the GPU). On an H100 this is a few minutes.

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
collator = DataCollatorForTokenClassification(tokenizer)
def metrics(p):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    preds = np.argmax(preds, axis=-1)
    gold = p.label_ids
    m = gold != -100
    return {'tok_acc': float((preds[m] == gold[m]).mean())}
args = TrainingArguments(
    output_dir='out', learning_rate=3e-5,
    per_device_train_batch_size=BS, per_device_eval_batch_size=BS * 2,
    num_train_epochs=4, weight_decay=0.01, warmup_ratio=0.06,
    bf16=USE_BF16, fp16=not USE_BF16, tf32=USE_BF16,
    optim='adamw_torch_fused', dataloader_num_workers=2,
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model='tok_acc', greater_is_better=True,
    logging_steps=50, report_to='none')
trainer = Trainer(model=model, args=args, train_dataset=tok_ds['train'],
                  eval_dataset=tok_ds['test'], data_collator=collator,
                  processing_class=tokenizer, compute_metrics=metrics)
trainer.train()
trainer.save_model('out_model'); tokenizer.save_pretrained('out_model')
print('final dev token-accuracy above is the sanity signal; the real lemma number is next.')

## 6 · Export to ONNX (fp32)
We ship **fp32**. Per-tensor int8 *dynamic* quantization collapses this ~9k-class head to the majority class (an outlier blows up the single weight scale). Production should use **per-channel** quantization (`quantize_dynamic(..., per_channel=True)`) or exclude the classifier MatMul, and validate against the dev score below before shipping.

In [ ]:
import os
!optimum-cli export onnx --model out_model --task token-classification onnx_fp32
onnx_path = 'onnx_fp32/model.onnx'
for f in sorted(os.listdir('onnx_fp32')):
    print(f, os.path.getsize(os.path.join('onnx_fp32', f)) // 1024, 'KB')

## 6b · In-Colab sanity — dev lemma accuracy (self-contained)
Decode the predicted edit-tree per token and score lemma vs gold on the dev split. This is the number that matters — see it **before** downloading, so a broken/quantized head can't slip through. (Mirrors the local torch-free eval exactly.)

In [ ]:
import numpy as np, onnxruntime as ort, re, unicodedata
from tokenizers import Tokenizer
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
tkf = Tokenizer.from_file('onnx_fp32/tokenizer.json')
inames = {i.name for i in sess.get_inputs()}
def _nrm(s): return unicodedata.normalize('NFC', s)
def _clean(s): return re.sub(r'\d+$', '', unicodedata.normalize('NFC', s))
def apply_tree(t, w):
    if t[0] == 'keep': return w
    if t[0] == 'sub': return t[1]
    _, pl, sl, lf, rt = t
    if pl + sl > len(w): return None
    p = apply_tree(lf, w[:pl]); s = apply_tree(rt, w[len(w) - sl:] if sl else '')
    if p is None or s is None: return None
    return p + (w[pl:len(w) - sl] if sl else w[pl:]) + s
dev = read_jsonl(DATA / 'dev.jsonl')
na = nu = oa = ou = 0
for srow in dev:
    forms = srow['tokens']; enc = tkf.encode(forms, is_pretokenized=True)
    feed = {'input_ids': np.array([enc.ids], dtype=np.int64)}
    if 'attention_mask' in inames:
        feed['attention_mask'] = np.array([enc.attention_mask], dtype=np.int64)
    lg = sess.run(None, feed)[0][0]
    first = {}
    for pos, wid in enumerate(enc.word_ids):
        if wid is not None and wid not in first: first[wid] = pos
    for wi, form in enumerate(forms):
        if not srow['scored'][wi]: continue
        na += 1; un = not srow['seen'][wi]; nu += un
        pos = first.get(wi); pred = _nrm(form)
        if pos is not None:
            out = apply_tree(json.loads(labels[int(np.argmax(lg[pos]))]), _nrm(form))
            if out is not None: pred = out
        if _clean(pred) == srow['lemmas'][wi]: oa += 1; ou += un
print(f'DEV lemma — all {oa/na:.1%}  UNSEEN {ou/nu:.1%}   '
      f'(beat: stanza 62.8% unseen; pure-Python 40.3%)')

## 7 · Package + download `spike_model.zip`

In [ ]:
import shutil
pathlib.Path('ship').mkdir(exist_ok=True)
shutil.copy(onnx_path, 'ship/model.onnx')
shutil.copy('onnx_fp32/tokenizer.json', 'ship/tokenizer.json')
shutil.copy(str(DATA / 'labels.json'), 'ship/labels.json')
shutil.make_archive('spike_model', 'zip', 'ship')
print('model.onnx', os.path.getsize('ship/model.onnx') // (1024 * 1024), 'MB')
files.download('spike_model.zip')

## Next
The **DEV lemma** number from 6b is the answer (unseen > 62.8% = we beat stanza). Send back `spike_model.zip` and I'll confirm it locally with the real pyaegean decode.
```
pip install onnxruntime tokenizers numpy
python eval_spike.py --model model.onnx --tokenizer tokenizer.json \
                     --labels data/labels.json --dev data/dev.jsonl
```